1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
 - 토큰수 초과로 답변을 생성하지 못할 수 있고
 - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을때, 벡터 db 유사도 검색
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달

# 1. 문서의 내용을 읽는다


In [ ]:
%pip install python-docx

In [ ]:
from docx import Document

document = Document('./tax.docx')
print(f'document == {dir(document)}')
full_text = ''

for index, paragraph in enumerate(document.paragraphs):
    print(f'paragraph == {paragraph.text}')
    full_text += f'{paragraph.text}\n'

# 2.문서를 쪼갠다

In [ ]:
%pip install tiktoken

In [ ]:
import tiktoken

def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model("gpt-4o")
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    text_list = []
    
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i: i+chunk_size]
        decoded = encoder.decode(chunk)
        text_list.append(decoded)
    
    return text_list

chunk_list = split_text(full_text, 1500)
chunk_list

# 3.임베딩

In [9]:
%pip install chromadb


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import chromadb

chroma_client = chromadb.Client()

In [29]:
collection_name = 'tax_collection'
tax_collection = chroma_client.create_collection(collection_name)

InternalError: Collection [tax_collection] already exists

In [ ]:
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma

openai_embedding = UpstageEmbeddings(model='solar-embedding-1-large')


In [32]:
tax_collection2 = chroma_client.get_or_create_collection('tax_collection2', embedding_function=openai_embedding)
id_list = []
for index in range(len(chunk_list)):
                   id_list.append(f'{index}')

tax_collection2.add(documents=chunk_list, ids=id_list)

AttributeError: 'UpstageEmbeddings' object has no attribute 'name'